# GP02C Revision Ledger

Exercise revision changes and verification receipt states on CPU.

This source copy comes from **GP02C_02_Evidence_Freshness_Ledger_CPU_v1_2.ipynb**. It keeps the study code and removes historical cell outputs and platform metadata. Original receipts and candidate files stay with the sealed run archives. See the [evidence index](../docs/evidence-index.md) for recorded outcomes and reproduction requirements.

Run the public companion for the self-contained CPU checks. Running these historical cells requires the relevant Kaggle inputs and the earlier evidence files.

**Clear-text source release:** No embedded ZIP, Base64 or hex payloads are required to read the notebook source.


### Preserved execution evidence — source archive

- **Executed source:** `Paper/GP02/GP02C/gp02c-02.ipynb` in the author-provided `Paper (2)(2).zip` (10 November 2026).
- **Saved results:** 5 human-readable Jupyter output records restored from matching executed cells.
- **Source fidelity:** 5 identical code cells; 0 proven value-equivalent readable source strings; 0 historical configuration cell(s) restored to align with archived execution.
- **Excluded display artifacts:** 0 active, encoded, or binary browser-download outputs were omitted; the source archive retains them. These are *not* replacements for the sealed experiment ZIPs.
- **Reproduction distinction:** Saved historical output does **not** mean this notebook has been rerun in the current environment. The original experiment may require Kaggle mounts, GPUs, offline wheels and authorised archives. The additional execution guide above details the prerequisites.


## How to execute or reproduce this implementation stage

**Execution mode:** CPU.  
**Prerequisites:** Kaggle Python environment and the CPU-only notebook dependencies.

**Steps:** Execute cells in order (including writefile cells) to create the ledger module and run state-transition acceptance tests. Compare PASS summaries.

**Reproduction boundary:** Runnable mechanism study; no Gemma inference required.

> **Historical evidence:** This release retains implementation source but removes historical cell outputs. Results from stochastic GPU runs are not guaranteed to repeat byte-for-byte. Use the sealed evidence records and the E17 reviewer companion to reconstruct accepted historical outcomes.


# GP02C-02 v1.2 — Evidence-Freshness Ledger (CPU-only)

**Research:** *When Passing Tests Become Stale: Evidence Freshness in Gemma 4 Software Repair Agents*  
**Stage:** GP02C-02 corrective CPU release (**56 deterministic tests**).  
**Runtime:** CPU / No Accelerator; **zero Gemma/GPU calls**, no protected tests and no 36-task evaluation.

**Review corrections incorporated:** A late workspace mutation cannot be approved using an earlier pass; the final permitted check is evaluated before rejecting; gate and delivery identities bind the entire relevant-workspace fingerprint **and** production patch; timeout and infrastructure outcomes remain separate; retries need matching configuration and environment identity; estimated-run overages are disqualified, never called a verified pass.

**IMPORTANT LIMITATION:** This is a **synthetic controller simulator**, not a trusted live execution adapter. A synchronous executor can physically overrun a simulated allowance. v1.2 records actual elapsed time and fails closed; real hard termination at 840s/900s and isolated immutable test snapshots remain **GP02C-03** acceptance requirements.

Original GP02B v2.7 development results and 36 held-out evaluation tasks remain untouched.


## 01 — Versioned evidence ledger and B/C gate

Store immutable visible-test receipts separately from their **current/stale** qualification, detect out-of-band source changes, keep observation and controller decisions apart, and tie approval to the exact production patch, full relevant-workspace SHA-256 and revision. A/B/C use identical passive instrumentation; B and C use the same final gate.


In [1]:
%%writefile gp02c_ledger.py
"""GP02C-02: CPU-only evidence-freshness ledger and B/C controller prototype.

This module does NOT run shell commands, use a model, access hidden evaluators,
or integrate with GP02B. A trusted adapter supplies visible-test results.
"""
from __future__ import annotations

from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable, Mapping, Sequence
import hashlib
import json
import os
import re
import time

SCHEMA = "gp02c.ledger.v1.2"
ARMS = {"A", "B", "C"}
PROVENANCES = {"agent", "controller_recheck", "final_gate", "common_initial"}
EXCLUDED_DIRS = frozenset({
    ".git", ".venv", "venv", "__pycache__", ".pytest_cache", ".mypy_cache",
    ".ruff_cache", ".tox", ".nox", ".cache", "node_modules", ".ipynb_checkpoints",
    ".gp02c_results", "__ledger_output__", "htmlcov", ".coverage-data",
})
EXCLUDED_NAMES = frozenset({".coverage", ".DS_Store", "coverage.xml"})
EXCLUDED_SUFFIXES = frozenset({".pyc", ".pyo"})
ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,120}$")


def canonical_hash(obj: object) -> str:
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=True,
                                      separators=(",", ":"), default=str).encode()).hexdigest()


def utc_timestamp() -> str:
    return datetime.now(timezone.utc).isoformat()


def environment_fingerprint(config: Mapping[str, object]) -> str:
    """Hash explicit environment identity supplied by the trusted harness.

    In live integration, include installed wheel/source hashes and relevant runtime
    settings, not just package names, and refresh after environment modifications.
    """
    if not config:
        raise ValueError("Nonempty trusted environment manifest required")
    return canonical_hash(dict(config))


@dataclass(frozen=True)
class Snapshot:
    sha256: str
    files: int
    entries: tuple[tuple[str, str, str], ...]


class WorkspaceFingerprinter:
    """Conservative all-files policy, except a fixed generated-artifact exclusion list.

    Includes untracked source and config; file additions, deletions and symlink
    targets affect the fingerprint. NEVER walk an evaluator/protected workspace.
    """
    def __init__(self, root: str | Path):
        self.root = Path(root).resolve(strict=True)
        if not self.root.is_dir():
            raise ValueError("Workspace root must be a directory")

    def snapshot(self) -> Snapshot:
        entries: list[tuple[str, str, str]] = []
        for base, dirs, files in os.walk(self.root, followlinks=False):
            basedir = Path(base)
            # Symlink directories are recorded as links, never dereferenced.
            for name in list(dirs):
                full = basedir / name
                if name in EXCLUDED_DIRS:
                    dirs.remove(name)
                elif full.is_symlink():
                    dirs.remove(name)
                    rel = full.relative_to(self.root).as_posix()
                    entries.append((rel, "L", hashlib.sha256(os.readlink(full).encode()).hexdigest()))
            for name in files:
                if name in EXCLUDED_NAMES or Path(name).suffix in EXCLUDED_SUFFIXES:
                    continue
                full = basedir / name
                rel = full.relative_to(self.root).as_posix()
                if full.is_symlink():
                    entries.append((rel, "L", hashlib.sha256(os.readlink(full).encode()).hexdigest()))
                elif full.is_file():
                    h = hashlib.sha256()
                    with full.open("rb") as f:
                        for block in iter(lambda: f.read(256 * 1024), b""):
                            h.update(block)
                    entries.append((rel, "F", h.hexdigest()))
        entries.sort()
        return Snapshot(canonical_hash(entries), len(entries), tuple(entries))


@dataclass(frozen=True)
class VisibleCheck:
    check_id: str
    command: tuple[str, ...]
    definition_sha256: str
    reserve_seconds: float = 0.0
    origin: str = "visible_repo"
    working_dir: str = "."
    test_selection: str = ""
    expected_collected: int | None = None
    pinned_environment_sha256: str | None = None

    def __post_init__(self):
        if not ID_RE.fullmatch(self.check_id):
            raise ValueError("Unsafe visible check identifier")
        if self.origin != "visible_repo":
            raise ValueError("Only trusted visible-repository checks are allowed")
        if not self.command or not all(isinstance(x, str) and x for x in self.command):
            raise ValueError("Nonempty command tokens required")
        if not re.fullmatch(r"[0-9a-f]{64}", self.definition_sha256):
            raise ValueError("Check definition must have SHA-256 identity")
        if self.reserve_seconds < 0:
            raise ValueError("Negative estimate")
        if (Path(self.working_dir).is_absolute() or
            any(x == ".." for x in Path(self.working_dir).parts)):
            raise ValueError("Working directory must be workspace-relative")
        if self.expected_collected is not None and self.expected_collected < 1:
            raise ValueError("Expected collection must be positive")
        if self.pinned_environment_sha256 is not None and not re.fullmatch(
            r"[0-9a-f]{64}", self.pinned_environment_sha256):
            raise ValueError("Bad pinned environment fingerprint")

    @property
    def command_sha256(self) -> str:
        return canonical_hash(self.command)


@dataclass(frozen=True)
class VisibleManifest:
    checks: tuple[VisibleCheck, ...]
    strict: bool = False

    def __post_init__(self):
        ids = [c.check_id for c in self.checks]
        if not ids or len(ids) != len(set(ids)):
            raise ValueError("Manifest needs at least one unique selected visible check")
        if self.strict and any(c.expected_collected is None or
                               c.pinned_environment_sha256 is None or
                               not c.test_selection for c in self.checks):
            raise ValueError("Strict manifests must pin collection, environment, and selection")

    def get(self, check_id: str) -> VisibleCheck:
        for check in self.checks:
            if check.check_id == check_id:
                return check
        raise ValueError("Unselected or protected check is not allowed")

    @property
    def sha256(self) -> str:
        return canonical_hash([asdict(c) for c in self.checks])


@dataclass(frozen=True)
class TestObservation:
    exit_code: int | None
    tests_collected: int | None
    tests_passed: int | None
    tests_failed: int | None
    tests_errored: int | None
    elapsed_seconds: float
    stdout: str = ""
    stderr: str = ""
    timed_out: bool = False
    setup_error: bool = False
    tests_skipped: int | None = 0

    def __post_init__(self):
        if self.elapsed_seconds < 0:
            raise ValueError("Negative duration")
        for name in ("tests_collected", "tests_passed", "tests_failed", "tests_errored", "tests_skipped"):
            v = getattr(self, name)
            if v is not None and v < 0:
                raise ValueError(f"Negative test count: {name}")

    @property
    def observed_outcome(self) -> str:
        if self.timed_out or self.setup_error or self.exit_code is None:
            return "ERROR"
        counts = (self.tests_collected, self.tests_passed,
                  self.tests_failed, self.tests_errored, self.tests_skipped)
        if any(c is None for c in counts):
            return "ERROR"
        total, passed, failed, errored, skipped = counts
        if total < 1 or passed + failed + errored + skipped != total:
            return "ERROR"
        if errored > 0:
            return "ERROR"
        if failed > 0 and self.exit_code != 0:
            return "FAIL"
        if failed > 0 and self.exit_code == 0:
            return "ERROR"  # contradictory successful process
        if self.exit_code != 0:
            return "ERROR"  # e.g. missing dependency / collection failure
        if passed < 1:
            return "ERROR"  # all skipped / no executed positive check
        return "PASS"


@dataclass(frozen=True)
class Receipt:
    receipt_id: str
    run_id: str
    task_id: str
    arm: str
    seed: int
    check_id: str
    command: tuple[str, ...]
    command_definition_sha256: str
    command_sha256: str
    working_dir: str
    test_selection: str
    expected_collected: int | None
    workspace_revision_at_start: int
    workspace_revision_at_end: int
    workspace_sha256_at_start: str
    workspace_sha256_at_end: str
    environment_sha256: str
    timestamp_start: str
    timestamp_end: str
    exit_code: int | None
    tests_collected: int | None
    tests_passed: int | None
    tests_failed: int | None
    tests_errored: int | None
    tests_skipped: int | None
    qualification_error: str | None
    stdout_sha256: str
    stderr_sha256: str
    provenance: str
    observed_outcome: str
    elapsed_seconds: float
    timed_out: bool
    setup_error: bool
    test_completed_monotonic: float


@dataclass(frozen=True)
class ChangeEvent:
    action: str
    previous_revision: int
    new_revision: int
    changed: bool
    previous_sha256: str
    new_sha256: str
    edit_observed_monotonic: float


class EvidenceLedger:
    def __init__(self, root: str | Path, manifest: VisibleManifest,
                 *, run_id: str, task_id: str, arm: str, seed: int,
                 environment_sha256: str):
        if arm not in ARMS:
            raise ValueError("Unknown arm")
        for value in (run_id, task_id):
            if not ID_RE.fullmatch(value):
                raise ValueError("Unsafe run/task id")
        if not re.fullmatch(r"[0-9a-f]{64}", environment_sha256):
            raise ValueError("Expected environment SHA-256")
        self.fingerprinter = WorkspaceFingerprinter(root)
        self.manifest = manifest
        self.run_id = run_id
        self.task_id = task_id
        self.arm = arm
        self.seed = seed
        self.environment_sha256 = environment_sha256
        self.snapshot = self.fingerprinter.snapshot()
        self.revision = 0
        self.receipts: list[Receipt] = []
        self.changes: list[ChangeEvent] = []
        self.events: list[dict] = []
        self.unqualified_receipts: dict[str, str] = {}

    def observe_workspace(self, action: str) -> ChangeEvent:
        new = self.fingerprinter.snapshot()
        old = self.snapshot
        before = self.revision
        changed = new.sha256 != old.sha256
        if changed:
            self.revision += 1
            self.snapshot = new
        ev = ChangeEvent(action, before, self.revision, changed, old.sha256,
                         new.sha256, time.monotonic())
        self.changes.append(ev)
        return ev

    def set_environment(self, environment_sha256: str) -> bool:
        if not re.fullmatch(r"[0-9a-f]{64}", environment_sha256):
            raise ValueError("Expected environment SHA-256")
        changed = self.environment_sha256 != environment_sha256
        self.environment_sha256 = environment_sha256
        if changed:
            self.events.append({"type": "environment_changed", "timestamp": utc_timestamp()})
        return changed

    def status(self, receipt: Receipt) -> str:
        check = self.manifest.get(receipt.check_id)
        self.observe_workspace("status_sync")  # detect out-of-band source changes
        if self.latest(check.check_id) is not receipt:
            return "STALE"
        if (receipt.workspace_revision_at_start != receipt.workspace_revision_at_end
            or receipt.workspace_sha256_at_start != receipt.workspace_sha256_at_end
            or receipt.workspace_revision_at_end != self.revision
            or receipt.workspace_sha256_at_end != self.snapshot.sha256
            or receipt.environment_sha256 != self.environment_sha256
            or receipt.command_definition_sha256 != check.definition_sha256
            or receipt.command_sha256 != check.command_sha256
            or receipt.working_dir != check.working_dir
            or receipt.test_selection != check.test_selection
            or (check.pinned_environment_sha256 is not None and
                check.pinned_environment_sha256 != self.environment_sha256)):

            return "STALE"
        if receipt.qualification_error or receipt.receipt_id in self.unqualified_receipts:
            return "ERROR"
        return {"PASS": "CURRENT_PASS", "FAIL": "CURRENT_FAIL"}.get(
            receipt.observed_outcome, "ERROR")

    def latest(self, check_id: str) -> Receipt | None:
        self.manifest.get(check_id)
        for receipt in reversed(self.receipts):
            if receipt.check_id == check_id:
                return receipt
        return None

    def current_status(self, check_id: str) -> str:
        r = self.latest(check_id)
        return self.status(r) if r else "NOT_EXECUTED"

    def selected_current_passes(self) -> tuple[str, ...]:
        return tuple(c.check_id for c in self.manifest.checks
                     if self.current_status(c.check_id) == "CURRENT_PASS")

    def record_check(self, check_id: str, provenance: str,
                     executor: Callable[[VisibleCheck], TestObservation]) -> Receipt:
        if provenance not in PROVENANCES:
            raise ValueError("Invalid provenance")
        check = self.manifest.get(check_id)
        self.observe_workspace("before_visible_check")
        start = self.snapshot
        rev_start = self.revision
        started = utc_timestamp()
        outcome = executor(check)
        if not isinstance(outcome, TestObservation):
            raise TypeError("Trusted visible executor must return TestObservation")
        ended = utc_timestamp()
        self.observe_workspace("after_visible_check")
        receipt = Receipt(
            receipt_id=f"visible-{len(self.receipts)+1:06d}",
            run_id=self.run_id, task_id=self.task_id, arm=self.arm, seed=self.seed,
            check_id=check.check_id, command=check.command,
            command_definition_sha256=check.definition_sha256,
            command_sha256=check.command_sha256,
            working_dir=check.working_dir, test_selection=check.test_selection,
            expected_collected=check.expected_collected,
            workspace_revision_at_start=rev_start,
            workspace_revision_at_end=self.revision,
            workspace_sha256_at_start=start.sha256,
            workspace_sha256_at_end=self.snapshot.sha256,
            environment_sha256=self.environment_sha256,
            timestamp_start=started, timestamp_end=ended,
            exit_code=outcome.exit_code, tests_collected=outcome.tests_collected,
            tests_passed=outcome.tests_passed, tests_failed=outcome.tests_failed,
            tests_errored=outcome.tests_errored,
            tests_skipped=outcome.tests_skipped,
            qualification_error=(
                "UNEXPECTED_COLLECTION" if check.expected_collected is not None
                and outcome.tests_collected != check.expected_collected else
                "ENVIRONMENT_MISMATCH" if check.pinned_environment_sha256 is not None
                and self.environment_sha256 != check.pinned_environment_sha256 else None),
            stdout_sha256=hashlib.sha256(outcome.stdout.encode()).hexdigest(),
            stderr_sha256=hashlib.sha256(outcome.stderr.encode()).hexdigest(),
            provenance=provenance, observed_outcome=outcome.observed_outcome,
            elapsed_seconds=outcome.elapsed_seconds, timed_out=outcome.timed_out,
            setup_error=outcome.setup_error,
            test_completed_monotonic=time.monotonic())
        # A stale label is not a regression. Record an observed pass-to-fail
        # transition only when both tests completed on stable, different source
        # revisions with identical environment/command identities.
        prior_pass = next((old for old in reversed(self.receipts)
                           if old.check_id == check.check_id
                           and old.observed_outcome == "PASS"
                           and old.qualification_error is None
                           and old.workspace_revision_at_start == old.workspace_revision_at_end
                           and old.workspace_sha256_at_start == old.workspace_sha256_at_end), None)
        self.receipts.append(receipt)
        if (prior_pass is not None and receipt.qualification_error is None
            and receipt.observed_outcome == "FAIL"
            and receipt.workspace_revision_at_start == receipt.workspace_revision_at_end
            and receipt.workspace_sha256_at_start == receipt.workspace_sha256_at_end
            and prior_pass.workspace_revision_at_end < receipt.workspace_revision_at_end
            and prior_pass.environment_sha256 == receipt.environment_sha256
            and prior_pass.command_definition_sha256 == receipt.command_definition_sha256
            and prior_pass.command_sha256 == receipt.command_sha256
            and not any(e.get("type") == "OBSERVED_PASS_TO_FAIL_AFTER_EDIT"
                        and e.get("prior_receipt_id") == prior_pass.receipt_id
                        for e in self.events)):
            edits = [e for e in self.changes if e.changed and
                     prior_pass.workspace_revision_at_end < e.new_revision <= receipt.workspace_revision_at_end]
            invalidating_edit = edits[0] if edits else None
            self.events.append({"type": "OBSERVED_PASS_TO_FAIL_AFTER_EDIT",
                                "check_id": check.check_id,
                                "prior_receipt_id": prior_pass.receipt_id,
                                "failure_receipt_id": receipt.receipt_id,
                                "prior_revision": prior_pass.workspace_revision_at_end,
                                "failure_revision": receipt.workspace_revision_at_end,
                                "invalidating_edit_observed_monotonic": (
                                    invalidating_edit.edit_observed_monotonic if invalidating_edit else None),
                                "failure_observed_monotonic": receipt.test_completed_monotonic,
                                "edit_to_detection_seconds": (
                                    max(0.0, receipt.test_completed_monotonic - invalidating_edit.edit_observed_monotonic)
                                    if invalidating_edit else None)})
        return receipt

    def disqualify_receipt(self, receipt: Receipt, reason: str) -> None:
        """Fail closed on controller-budget violations without mutating raw observations."""
        self.unqualified_receipts[receipt.receipt_id] = reason
        self.events.append({"type": "RECEIPT_UNQUALIFIED", "receipt_id": receipt.receipt_id,
                            "reason": reason})

    def export(self) -> dict:
        # Refresh revision *before* writing any status rows or reporting a hash.
        self.observe_workspace("export_preflight")
        return {
            "schema": SCHEMA, "run_id": self.run_id, "task_id": self.task_id,
            "arm": self.arm, "seed": self.seed, "manifest_sha256": self.manifest.sha256,
            "workspace_revision": self.revision, "workspace_sha256": self.snapshot.sha256,
            "environment_sha256": self.environment_sha256,
            "receipts": [{**asdict(r), "status": self.status(r),
                          "controller_disqualification": self.unqualified_receipts.get(r.receipt_id)}
                         for r in self.receipts],
            "changes": [asdict(e) for e in self.changes],
            "events": list(self.events),
        }


@dataclass
class SharedBudget:
    max_wall_seconds: float = 900.0
    max_tool_calls: int = 60
    max_model_calls: int = 60
    used_wall_seconds: float = 0.0
    used_tool_calls: int = 0
    used_model_calls: int = 0
    used_test_runs: int = 0
    final_reserve_seconds: float = 60.0

    @property
    def remaining_wall(self) -> float:
        return max(0.0, self.max_wall_seconds - self.used_wall_seconds)

    def can_run_check(self, check: VisibleCheck, *, protect_final_reserve: bool = False) -> bool:
        reserve = self.final_reserve_seconds if protect_final_reserve else 0.0
        return (self.used_tool_calls < self.max_tool_calls
                and self.remaining_wall > 0
                and self.remaining_wall >= check.reserve_seconds + reserve)

    @property
    def remaining_repair_wall(self) -> float:
        return max(0.0, self.max_wall_seconds - self.final_reserve_seconds - self.used_wall_seconds)

    @property
    def over_limit(self) -> bool:
        return (self.used_wall_seconds > self.max_wall_seconds
                or self.used_tool_calls > self.max_tool_calls
                or self.used_model_calls > self.max_model_calls)

    def charge_model(self, elapsed_seconds: float = 0.0) -> None:
        if self.used_model_calls >= self.max_model_calls:
            raise RuntimeError("Model budget exhausted")
        self.used_model_calls += 1
        self.used_wall_seconds += elapsed_seconds

    def charge_tool(self, elapsed_seconds: float = 0.0) -> None:
        if self.used_tool_calls >= self.max_tool_calls:
            raise RuntimeError("Tool budget exhausted")
        self.used_tool_calls += 1
        self.used_wall_seconds += elapsed_seconds

    def charge_visible_test(self, elapsed_seconds: float) -> None:
        # Executing a selected check also consumes one shared tool invocation.
        self.charge_tool(elapsed_seconds)
        self.used_test_runs += 1

    def summary(self) -> dict:
        return {"used_wall_seconds": self.used_wall_seconds,
                "remaining_wall_seconds": self.remaining_wall,
                "used_tool_calls": self.used_tool_calls,
                "used_model_calls": self.used_model_calls,
                "used_test_runs": self.used_test_runs}


@dataclass(frozen=True)
class GateDecision:
    accepted: bool
    status: str
    checks: dict[str, str]
    attempts: int
    accepted_patch_sha256: str | None = None
    accepted_workspace_sha256: str | None = None
    accepted_revision: int | None = None


class VerificationController:
    """Trusted wrapper prototype. Same gate implementation for B and C.

    Executor is a trusted *visible* check adapter; NEVER pass hidden evaluator
    results. This prototype uses deterministic duration accounting. Live GP02C-03
    must enforce wall time and tool/model counters on the actual worker.
    """
    def __init__(self, ledger: EvidenceLedger, budget: SharedBudget,
                 executor: Callable[[VisibleCheck], TestObservation],
                 production_patch_sha256: Callable[[], str],
                 *, max_rechecks_per_edit: int = 4,
                 max_rechecks_per_run: int = 12):
        self.ledger = ledger
        self.budget = budget
        self.executor = executor
        self.production_patch_sha256 = production_patch_sha256
        if max_rechecks_per_edit < 0 or max_rechecks_per_run < 0:
            raise ValueError("Negative auto-recheck limit")
        self.max_rechecks_per_edit = max_rechecks_per_edit
        self.max_rechecks_per_run = max_rechecks_per_run
        self.auto_rechecks_used = 0

    def _patch_digest(self) -> str:
        digest = self.production_patch_sha256()
        if not isinstance(digest, str) or not re.fullmatch(r"[0-9a-f]{64}", digest):
            raise ValueError("Trusted production-patch capture must return a SHA-256 digest")
        return digest

    def _run_check(self, check_id: str, provenance: str) -> Receipt | None:
        check = self.ledger.manifest.get(check_id)
        proactive = provenance == "controller_recheck"
        if not self.budget.can_run_check(check, protect_final_reserve=proactive):
            self.ledger.events.append({"type": "UNVERIFIED_BUDGET", "check_id": check_id,
                                       "provenance": provenance})
            return None
        # CPU contract only: the caller-supplied executor is synchronous and NOT
        # cancellable. It must be replaced in GP02C-03 by a hard-deadline trusted
        # subprocess adapter, using a monotonic 840/900-second cutoff.
        allowed_seconds = (self.budget.remaining_repair_wall if proactive
                           else self.budget.remaining_wall)
        def recorded_executor(c: VisibleCheck) -> TestObservation:
            try:
                result = self.executor(c)
            except Exception as e:
                result = TestObservation(None, None, None, None, None,
                                         elapsed_seconds=0.0, setup_error=True)
                self.ledger.events.append({"type": "executor_error", "check_id": check_id,
                                           "error_type": type(e).__name__})
            self.budget.charge_visible_test(result.elapsed_seconds)
            return result
        receipt = self.ledger.record_check(check_id, provenance, recorded_executor)
        if (receipt.elapsed_seconds > allowed_seconds + 1e-9 or
            self.budget.over_limit or
            (proactive and self.budget.remaining_repair_wall <= 0
             and self.budget.used_wall_seconds > self.budget.max_wall_seconds-self.budget.final_reserve_seconds + 1e-9)):
            self.ledger.disqualify_receipt(receipt, "ACTUAL_RUNTIME_EXCEEDED_ALLOWED_DEADLINE")
            self.ledger.events.append({"type": "UNVERIFIED_BUDGET",
                                       "check_id": check_id, "provenance": provenance,
                                       "reason": "actual_runtime_overrun_requires_live_cancellation"})
        return receipt

    def observe_agent_action(self, kind: str, action: Callable[[], object],
                             *, elapsed_seconds: float = 0.0) -> dict:
        """Wrap any tool action: read/edit/write/shell; no mandatory read gate.

        Charge the agent action itself; C post-edit checks are additional charged
        calls. Exceptions preserve before/after snapshots and are re-raised.
        """
        if (self.budget.used_tool_calls >= self.budget.max_tool_calls
            or self.budget.remaining_wall <= (
                self.budget.final_reserve_seconds if self.ledger.arm in {"B", "C"} else 0)):
            raise RuntimeError("Shared agent-tool budget exhausted or reserved for final validation")
        self.ledger.observe_workspace(f"before_{kind}")
        previously_passing = self.ledger.selected_current_passes()
        exc: BaseException | None = None
        try:
            result = action()
        except BaseException as error:
            exc = error
            result = None
        finally:
            # Conservative: one tool invocation is billed even if it errors.
            self.budget.charge_tool(elapsed_seconds)
            change = self.ledger.observe_workspace(f"after_{kind}")
        reruns: list[dict[str, str]] = []
        if change.changed and self.ledger.arm == "C":
            for position, check_id in enumerate(previously_passing):
                if (position >= self.max_rechecks_per_edit or
                    self.auto_rechecks_used >= self.max_rechecks_per_run):
                    status = "UNVERIFIED_RECHECK_LIMIT"
                    self.ledger.events.append({"type": status, "check_id": check_id})
                else:
                    self.auto_rechecks_used += 1
                    receipt = self._run_check(check_id, "controller_recheck")
                    status = ("UNVERIFIED_BUDGET" if receipt is None or
                              receipt.receipt_id in self.ledger.unqualified_receipts or
                              self.budget.over_limit else self.ledger.status(receipt))
                reruns.append({"check_id": check_id, "status": status})
        feedback = {"changed": change.changed, "revision": change.new_revision,
                    "stale_selected_checks": list(previously_passing) if change.changed else [],
                    "rechecks": reruns}
        if exc is not None:
            raise exc
        return {"result": result, "feedback": feedback}

    def agent_visible_feedback(self, result: dict) -> dict:
        """Bounded and exclusively derived from allowlisted visible-check IDs.

        Never includes test stdout, stderr, hidden tests, reference patch,
        tool outputs or the model context; the integration must preserve this.
        """
        feedback = result["feedback"]
        return {"schema": "gp02c.feedback.v1", "revision": feedback["revision"],
                "changed": feedback["changed"],
                "stale": feedback["stale_selected_checks"][:32],
                "rechecks": feedback["rechecks"][:32]}

    def final_gate(self) -> GateDecision:
        if self.ledger.arm == "A":
            raise ValueError("Ordinary baseline A has no mandatory final gate")
        # Same B/C gate. Each decision revalidates all selected statuses *after*
        # the last source sync and is bound to the workspace SHA, revision,
        # production patch SHA and frozen manifest SHA.
        self.ledger.observe_workspace("final_gate_entry")
        initial_patch = self._patch_digest()
        attempts = 0
        max_attempts = len(self.ledger.manifest.checks) * 3

        def statuses_now() -> dict[str, str]:
            self.ledger.observe_workspace("final_gate_status_sync")
            return {c.check_id: self.ledger.current_status(c.check_id)
                    for c in self.ledger.manifest.checks}

        while True:
            statuses = statuses_now()
            # Do not declare exhaustion before reviewing the final allowed result.
            if all(s == "CURRENT_PASS" for s in statuses.values()):
                self.ledger.observe_workspace("final_gate_patch_sync")
                # The final sync may itself reveal a mutation. Recompute *after* it.
                statuses = statuses_now()
                workspace = self.ledger.snapshot.sha256
                revision = self.ledger.revision
                if self._patch_digest() != initial_patch:
                    return GateDecision(False, "PATCH_CHANGED_DURING_GATE", statuses, attempts)
                # One additional consistency check after the patch capture.
                self.ledger.observe_workspace("final_gate_approval_sync")
                statuses = statuses_now()
                if (all(s == "CURRENT_PASS" for s in statuses.values()) and
                    self.ledger.snapshot.sha256 == workspace and
                    self.ledger.revision == revision and
                    self._patch_digest() == initial_patch and
                    not self.budget.over_limit and self.budget.remaining_wall > 0):
                    return GateDecision(True, "VERIFIED_VISIBLE", statuses, attempts,
                                        accepted_patch_sha256=initial_patch,
                                        accepted_workspace_sha256=workspace,
                                        accepted_revision=revision)
            if self.budget.over_limit or self.budget.remaining_wall <= 0:
                return GateDecision(False, "UNVERIFIED_BUDGET", statuses, attempts)
            if any(s in {"CURRENT_FAIL", "ERROR"} for s in statuses.values()):
                return GateDecision(False, "VISIBLE_CHECK_FAILED", statuses, attempts)
            pending = [cid for cid, s in statuses.items()
                       if s in {"STALE", "NOT_EXECUTED"}]
            if not pending:
                return GateDecision(False, "UNVERIFIED", statuses, attempts)
            if attempts >= max_attempts:
                return GateDecision(False, "UNVERIFIED_RECHECK_LIMIT", statuses, attempts)
            receipt = self._run_check(pending[0], "final_gate")
            if receipt is None or receipt.receipt_id in self.ledger.unqualified_receipts:
                return GateDecision(False, "UNVERIFIED_BUDGET", statuses_now(), attempts)
            attempts += 1


# Protected results enter ONLY after the agent and controller are finished.
# This function is an offline analyst's scorer, not an agent-visible tool.
@dataclass(frozen=True)
class AttemptRecord:
    task_id: str
    arm: str
    seed: int
    attempt_index: int
    termination: str  # SUBMITTED | TIMEOUT | STOPPED | GATE_REJECTED | INFRA_ERROR
    explicit_submit: bool
    patch_present: bool
    final_patch_sha256: str | None
    final_workspace_revision: int
    protected_result: str  # PASS | FAIL | NOT_RUN | INFRA_ERROR
    gate_decision: GateDecision | None = None
    infrastructure_retry_of: int | None = None
    final_workspace_sha256: str | None = None
    environment_sha256: str | None = None
    configuration_sha256: str | None = None
    retry_evidence_id: str | None = None

    def __post_init__(self):
        if self.arm not in ARMS or self.attempt_index < 1:
            raise ValueError("Invalid attempt identity")
        if self.protected_result not in {"PASS", "FAIL", "NOT_RUN", "INFRA_ERROR"}:
            raise ValueError("Unsupported protected grading status")
        if self.termination not in {"SUBMITTED", "TIMEOUT", "STOPPED", "GATE_REJECTED", "INFRA_ERROR"}:
            raise ValueError("Unsupported termination status")
        if self.patch_present:
            if not self.final_patch_sha256 or not re.fullmatch(r"[0-9a-f]{64}", self.final_patch_sha256):
                raise ValueError("Captured production patch needs a trusted SHA-256")
        elif self.final_patch_sha256 is not None:
            raise ValueError("Absent production patch must not have a digest")
        for name in ("final_workspace_sha256", "environment_sha256", "configuration_sha256"):
            v = getattr(self, name)
            if v is not None and not re.fullmatch(r"[0-9a-f]{64}", v):
                raise ValueError(f"Invalid {name}")


@dataclass(frozen=True)
class AttemptAssessment:
    protected_verification: str
    verified_delivery: bool
    delivery_reason: str
    termination: str
    infrastructure_fault: bool
    patch_present: bool


def assess_attempt(attempt: AttemptRecord) -> AttemptAssessment:
    """Offline-only: classify grading separately from attempted delivery.

    The trusted scorer must separately capture the final *production* patch,
    final relevant-workspace fingerprint and protected verifier result.
    """
    infra_fault = (attempt.termination == "INFRA_ERROR" or
                   attempt.protected_result == "INFRA_ERROR")
    protected = (attempt.protected_result if attempt.patch_present
                 else "INFRA_ERROR" if infra_fault else "NOT_RUN")
    def result(delivered: bool, why: str) -> AttemptAssessment:
        return AttemptAssessment(protected, delivered, why,
                                 attempt.termination, infra_fault, attempt.patch_present)
    if not attempt.patch_present:
        return result(False, "INFRASTRUCTURE_FAILURE" if infra_fault else "NO_PRODUCTION_PATCH")
    if not attempt.explicit_submit:
        return result(False, "NO_EXPLICIT_SUBMISSION")
    if attempt.termination != "SUBMITTED":
        return result(False, "NOT_DELIVERED")
    if attempt.arm in {"B", "C"}:
        gate = attempt.gate_decision
        if gate is None or not gate.accepted:
            return result(False, "FINAL_GATE_NOT_ACCEPTED")
        if (gate.accepted_patch_sha256 != attempt.final_patch_sha256 or
            gate.accepted_revision != attempt.final_workspace_revision or
            gate.accepted_workspace_sha256 != attempt.final_workspace_sha256):
            return result(False, "PATCH_OR_WORKSPACE_CHANGED_AFTER_GATE")
    if protected != "PASS":
        return result(False, "PROTECTED_VERIFICATION_NOT_PASS")
    return result(True, "VERIFIED_DELIVERY")


class AttemptRegister:
    """Keep every attempt. First attempt is primary unless *its* failure was
    independently classified as infrastructure and a documented retry exists.
    Never replace a genuine timeout or repair failure with a lucky rerun.
    """
    def __init__(self):
        self.attempts: list[AttemptRecord] = []

    def append(self, attempt: AttemptRecord) -> None:
        prior = [a for a in self.attempts if
                 (a.task_id, a.arm, a.seed) == (attempt.task_id, attempt.arm, attempt.seed)]
        if attempt.attempt_index != len(prior) + 1:
            raise ValueError("Attempt indices must be contiguous and append-only")
        if prior and attempt.infrastructure_retry_of != prior[-1].attempt_index:
            raise ValueError("Repeat must document preceding infrastructure retry")
        if prior and prior[-1].termination != "INFRA_ERROR":
            raise ValueError("Only infrastructure faults permit confirmatory replacement")
        if prior:
            previous = prior[-1]
            if not attempt.retry_evidence_id or not ID_RE.fullmatch(attempt.retry_evidence_id):
                raise ValueError("Infrastructure retry requires recorded evidence ID")
            if not all((attempt.environment_sha256, previous.environment_sha256,
                        attempt.configuration_sha256, previous.configuration_sha256)):
                raise ValueError("Infrastructure retry needs both frozen fingerprints")
            if (attempt.environment_sha256 != previous.environment_sha256 or
                attempt.configuration_sha256 != previous.configuration_sha256):
                raise ValueError("Infrastructure retry cannot change the frozen environment/config")
        self.attempts.append(attempt)

    def primary(self, task_id: str, arm: str, seed: int) -> AttemptRecord:
        matches = [a for a in self.attempts if
                   (a.task_id, a.arm, a.seed) == (task_id, arm, seed)]
        if not matches:
            raise KeyError("No attempt")
        return matches[-1]  # only documented infrastructure retries can follow


Writing gp02c_ledger.py


**Cell 01 observation criterion:** `gp02c_ledger.py` is created successfully. This is code generation, not an LLM run or a measured repair result.

## 02 — CPU-only contract regression tests (56 scenarios)

The 46 original v1.1 tests are retained and strengthened with ten v1.2 tests for late edits, final-run acceptance, over-budget disqualification, post-gate workspace mutations, infrastructure classification, retry evidence and consistent B/C decisions. These fixtures are **synthetic observations**, not live `pytest` runs.


In [2]:
%%writefile test_gp02c_ledger.py
"""Deterministic CPU-only contract probes. No LLM/GPU/competition task access."""
import hashlib
import json
from pathlib import Path
from tempfile import TemporaryDirectory
import unittest

from gp02c_ledger import (
    EvidenceLedger, SharedBudget, TestObservation, VerificationController,
    VisibleCheck, VisibleManifest, WorkspaceFingerprinter, environment_fingerprint,
    AttemptRecord, assess_attempt, AttemptRegister, GateDecision,
)

PASS = TestObservation(0, 1, 1, 0, 0, 0.5, stdout="1 passed")
FAIL = TestObservation(1, 1, 0, 1, 0, 0.6, stdout="1 failed")
ERROR = TestObservation(4, None, None, None, None, 0.2, setup_error=True)


class Fixtures(unittest.TestCase):
    def setUp(self):
        self.tmp = TemporaryDirectory(prefix="gp02c_02_")
        self.addCleanup(self.tmp.cleanup)
        self.root = Path(self.tmp.name)
        self.code = self.root / "app.py"
        self.code.write_text("VALUE = 1\n")
        self.env = environment_fingerprint({"python": "3.x", "wheel_sha256": "fixture-wheel-A"})
        self.check = VisibleCheck("x", ("pytest", "-q", "tests/test_x.py"),
                                  hashlib.sha256(b"visible-check-definition-x").hexdigest(), 0.4,
                                  working_dir=".", test_selection="tests/test_x.py",
                                  expected_collected=1, pinned_environment_sha256=self.env)
        self.check2 = VisibleCheck("y", ("pytest", "-q", "tests/test_y.py"),
                                   hashlib.sha256(b"visible-check-definition-y").hexdigest(), 0.4,
                                   working_dir=".", test_selection="tests/test_y.py",
                                   expected_collected=1, pinned_environment_sha256=self.env)
        self.manifest = VisibleManifest((self.check,), strict=True)

    def ledger(self, arm="C", manifest=None):
        return EvidenceLedger(self.root, manifest or self.manifest, run_id="fixture-001",
                              task_id="cpu_fixture", arm=arm, seed=20261105,
                              environment_sha256=self.env)

    def ctrl(self, arm="C", budget=None, executor=None, manifest=None):
        ledger = self.ledger(arm, manifest)
        return VerificationController(ledger, budget or SharedBudget(),
                                      executor or (lambda c: PASS),
                                      production_patch_sha256=lambda: hashlib.sha256(self.code.read_bytes()).hexdigest())

    def test_01_pass_then_edit_stales_without_regression(self):
        led = self.ledger()
        receipt = led.record_check("x", "agent", lambda _: PASS)
        self.assertEqual(led.status(receipt), "CURRENT_PASS")
        self.code.write_text("VALUE = 2\n")
        ev = led.observe_workspace("edit_file")
        self.assertTrue(ev.changed)
        self.assertEqual(ev.new_revision, 1)
        self.assertEqual(led.status(receipt), "STALE")
        self.assertEqual(receipt.observed_outcome, "PASS")

    def test_02_post_edit_pass_new_receipt(self):
        led = self.ledger()
        old = led.record_check("x", "agent", lambda _: PASS)
        self.code.write_text("VALUE = 2\n")
        new = led.record_check("x", "controller_recheck", lambda _: PASS)
        self.assertEqual(led.status(old), "STALE")
        self.assertEqual(led.status(new), "CURRENT_PASS")
        self.assertNotEqual(old.workspace_revision_at_end, new.workspace_revision_at_end)

    def test_03_pass_then_edit_then_fail_observed_regression(self):
        ctrl = self.ctrl()
        led = ctrl.ledger
        led.record_check("x", "agent", lambda _: PASS)
        result = ctrl.observe_agent_action("edit_file", lambda: self.code.write_text("VALUE = 2\n"))
        self.assertEqual(result["feedback"]["rechecks"], [{"check_id": "x", "status": "CURRENT_PASS"}])
        self.assertEqual(ctrl.budget.used_test_runs, 1)
        self.assertEqual(led.current_status("x"), "CURRENT_PASS")
        # Separate controller recheck explicitly fails when regression is visible.
        ctrl2 = self.ctrl(executor=lambda c: FAIL)
        ctrl2.ledger.record_check("x", "agent", lambda _: PASS)
        out = ctrl2.observe_agent_action("edit_file", lambda: self.code.write_text("VALUE = 3\n"))
        self.assertEqual(out["feedback"]["rechecks"][0]["status"], "CURRENT_FAIL")
        self.assertEqual(ctrl2.ledger.latest("x").observed_outcome, "FAIL")
        self.assertTrue(any(e["type"] == "OBSERVED_PASS_TO_FAIL_AFTER_EDIT"
                            for e in ctrl2.ledger.events))

    def test_04_rejected_or_noop_edit_does_not_advance_revision(self):
        ctrl = self.ctrl()
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        def rejected():
            raise PermissionError("restricted")
        with self.assertRaises(PermissionError):
            ctrl.observe_agent_action("edit_file", rejected)
        self.assertEqual(ctrl.ledger.revision, 0)
        self.assertEqual(ctrl.ledger.current_status("x"), "CURRENT_PASS")
        self.assertEqual(ctrl.budget.used_tool_calls, 1)

    def test_05_shell_mutation_detected(self):
        led = self.ledger()
        led.record_check("x", "agent", lambda _: PASS)
        self.code.write_text("VALUE = 5\n")
        self.assertEqual(led.current_status("x"), "STALE")
        self.assertEqual(led.revision, 1)

    def test_06_test_itself_mutates_code_invalid_receipt(self):
        led = self.ledger()
        def mutating(_):
            self.code.write_text("VALUE = 6\n")
            return PASS
        receipt = led.record_check("x", "agent", mutating)
        self.assertEqual(led.status(receipt), "STALE")
        self.assertEqual(receipt.observed_outcome, "PASS")

    def test_07_zero_collection_is_not_pass(self):
        led = self.ledger()
        receipt = led.record_check("x", "agent", lambda _: TestObservation(0, 0, 0, 0, 0, 0.1))
        self.assertEqual(led.status(receipt), "ERROR")

    def test_08_timeout_error_malformed_results_and_skipped(self):
        samples = [
            TestObservation(None, None, None, None, None, 0.4, timed_out=True),
            ERROR,
            TestObservation(0, None, None, None, None, 0.2),
            TestObservation(0, 1, 0, 0, 0, 0.1, tests_skipped=1),
            TestObservation(0, 1, 0, 1, 0, 0.1),
        ]
        for sample in samples:
            with self.subTest(observation=sample):
                led = self.ledger()
                r = led.record_check("x", "agent", lambda _: sample)
                self.assertEqual(led.status(r), "ERROR")

    def test_09_environment_change_invalidates_but_not_workspace_revision(self):
        led = self.ledger()
        r = led.record_check("x", "agent", lambda _: PASS)
        led.set_environment(environment_fingerprint({"python": "3.x", "wheel_sha256": "B"}))
        self.assertEqual(led.status(r), "STALE")
        self.assertEqual(led.revision, 0)

    def test_10_revert_bytes_cannot_resurrect_old_pass(self):
        led = self.ledger()
        r = led.record_check("x", "agent", lambda _: PASS)
        self.code.write_text("VALUE = 9\n")
        led.observe_workspace("modify")
        self.code.write_text("VALUE = 1\n")
        led.observe_workspace("revert")
        self.assertEqual(led.revision, 2)
        self.assertEqual(led.snapshot.sha256, r.workspace_sha256_at_end)
        self.assertEqual(led.status(r), "STALE")

    def test_11_no_prior_pass_no_auto_recheck(self):
        ctrl = self.ctrl()
        out = ctrl.observe_agent_action("write_file", lambda: self.code.write_text("VALUE = 11\n"))
        self.assertEqual(out["feedback"]["rechecks"], [])
        self.assertEqual(ctrl.budget.used_tool_calls, 1)
        self.assertEqual(ctrl.budget.used_test_runs, 0)

    def test_12_C_insufficient_budget_does_not_fake_pass(self):
        budget = SharedBudget(max_wall_seconds=0.5, max_tool_calls=3, final_reserve_seconds=0.0)
        ctrl = self.ctrl(budget=budget)
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        out = ctrl.observe_agent_action("write_file", lambda: self.code.write_text("VALUE = 12\n"),
                                        elapsed_seconds=0.2)
        self.assertEqual(out["feedback"]["rechecks"],
                         [{"check_id": "x", "status": "UNVERIFIED_BUDGET"}])
        self.assertEqual(ctrl.budget.used_test_runs, 0)
        self.assertEqual(ctrl.ledger.current_status("x"), "STALE")

    def test_13_B_final_gate_rechecks_after_edit(self):
        ctrl = self.ctrl(arm="B")
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        self.code.write_text("VALUE = 13\n")
        decision = ctrl.final_gate()
        self.assertTrue(decision.accepted)
        self.assertEqual(decision.status, "VERIFIED_VISIBLE")
        self.assertEqual(ctrl.ledger.latest("x").provenance, "final_gate")
        self.assertEqual(ctrl.budget.used_tool_calls, 1)

    def test_14_B_final_gate_fails_actual_failure(self):
        ctrl = self.ctrl(arm="B", executor=lambda _: FAIL)
        dec = ctrl.final_gate()
        self.assertFalse(dec.accepted)
        self.assertEqual(dec.status, "VISIBLE_CHECK_FAILED")
        self.assertEqual(ctrl.ledger.current_status("x"), "CURRENT_FAIL")

    def test_15_B_C_identical_final_gate(self):
        for arm in ("B", "C"):
            with self.subTest(arm=arm):
                ctrl = self.ctrl(arm=arm)
                dec = ctrl.final_gate()
                self.assertTrue(dec.accepted)
                self.assertEqual(dec.attempts, 1)
                self.assertEqual(ctrl.budget.used_test_runs, 1)

    def test_16_A_no_final_gate(self):
        ctrl = self.ctrl(arm="A")
        with self.assertRaises(ValueError):
            ctrl.final_gate()

    def test_17_protected_or_unlisted_check_blocked(self):
        with self.assertRaises(ValueError):
            VisibleCheck("hidden", ("pytest", "hidden.py"), "0" * 64,
                         origin="protected_evaluator")
        with self.assertRaises(ValueError):
            self.ledger().record_check("hidden", "agent", lambda _: PASS)

    def test_18_feedback_never_contains_private_outputs(self):
        secret = "PROTECTED_REFERENCE_PATCH: DO_NOT_EXPOSE"
        ctrl = self.ctrl(executor=lambda c: TestObservation(0, 1, 1, 0, 0, 0.3,
                                                             stdout=secret, stderr=secret))
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        result = ctrl.observe_agent_action("edit_file", lambda: self.code.write_text("VALUE = 18\n"))
        feedback = json.dumps(ctrl.agent_visible_feedback(result))
        self.assertNotIn(secret, feedback)
        self.assertEqual(ctrl.ledger.current_status("x"), "CURRENT_PASS")

    def test_19_no_mandatory_investigation_tool_call(self):
        for arm in ("A", "B", "C"):
            with self.subTest(arm=arm):
                ctrl = self.ctrl(arm=arm)
                outcome = ctrl.observe_agent_action("edit_file", lambda: self.code.write_text("VALUE = 19\n"))
                self.assertTrue(outcome["feedback"]["changed"] or ctrl.ledger.revision == 0)
                self.assertEqual(ctrl.budget.used_tool_calls, 1)

    def test_20_untracked_addition_deletion_and_exclusion(self):
        fp = WorkspaceFingerprinter(self.root)
        first = fp.snapshot()
        untracked = self.root / "helper.py"
        untracked.write_text("print('helper')\n")
        second = fp.snapshot()
        self.assertNotEqual(first.sha256, second.sha256)
        untracked.unlink()
        self.assertEqual(fp.snapshot().sha256, first.sha256)
        cache = self.root / "__pycache__"
        cache.mkdir()
        (cache / "app.pyc").write_bytes(b"ignored")
        self.assertEqual(fp.snapshot().sha256, first.sha256)

    def test_21_failure_supersedes_earlier_pass_same_revision(self):
        led = self.ledger()
        old = led.record_check("x", "agent", lambda _: PASS)
        new = led.record_check("x", "agent", lambda _: FAIL)
        self.assertEqual(led.status(old), "STALE")
        self.assertEqual(led.status(new), "CURRENT_FAIL")

    def test_22_B_final_budget_exhaustion(self):
        ctrl = self.ctrl(arm="B", budget=SharedBudget(max_wall_seconds=0.2))
        dec = ctrl.final_gate()
        self.assertFalse(dec.accepted)
        self.assertEqual(dec.status, "UNVERIFIED_BUDGET")
        self.assertEqual(ctrl.budget.used_test_runs, 0)

    def test_23_test_mutation_prevents_final_acceptance(self):
        def mutate(check):
            self.code.write_text(self.code.read_text() + "# touched\n")
            return PASS
        ctrl = self.ctrl(arm="B", executor=mutate)
        dec = ctrl.final_gate()
        self.assertFalse(dec.accepted)
        self.assertIn(dec.status, {"UNVERIFIED_RECHECK_LIMIT", "UNVERIFIED_BUDGET"})

    def test_24_agent_action_C_auto_rechecks_and_charges_shared_calls(self):
        ctrl = self.ctrl()
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        ctrl.observe_agent_action("run_command", lambda: self.code.write_text("VALUE = 24\n"),
                                  elapsed_seconds=2.0)
        self.assertEqual(ctrl.budget.used_tool_calls, 2)
        self.assertEqual(ctrl.budget.used_test_runs, 1)
        self.assertAlmostEqual(ctrl.budget.used_wall_seconds, 2.5)

    def test_25_B_does_not_proactively_recheck(self):
        ctrl = self.ctrl(arm="B")
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        ctrl.observe_agent_action("edit_file", lambda: self.code.write_text("VALUE = 25\n"))
        self.assertEqual(ctrl.budget.used_test_runs, 0)
        self.assertEqual(ctrl.ledger.current_status("x"), "STALE")

    def test_26_multiple_selected_checks_C_proactive_scope(self):
        manifest = VisibleManifest((self.check, self.check2))
        ctrl = self.ctrl(manifest=manifest)
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        ctrl.ledger.record_check("y", "agent", lambda _: FAIL)
        res = ctrl.observe_agent_action("write_file", lambda: self.code.write_text("VALUE = 26\n"))
        self.assertEqual([x["check_id"] for x in res["feedback"]["rechecks"]], ["x"])
        self.assertEqual(ctrl.ledger.current_status("y"), "STALE")

    def test_27_executor_exception_does_not_approve(self):
        def explode(_):
            raise FileNotFoundError("Missing test setup")
        ctrl = self.ctrl(arm="B", executor=explode)
        dec = ctrl.final_gate()
        self.assertFalse(dec.accepted)
        self.assertEqual(ctrl.ledger.current_status("x"), "ERROR")

    def test_29_test_actual_runtime_overrun_cannot_be_accepted(self):
        ctrl = self.ctrl(arm="B", budget=SharedBudget(max_wall_seconds=1.0),
                         executor=lambda c: TestObservation(0, 1, 1, 0, 0, 3.0))
        dec = ctrl.final_gate()
        self.assertFalse(dec.accepted)
        self.assertEqual(dec.status, "UNVERIFIED_BUDGET")
        self.assertEqual(ctrl.budget.used_test_runs, 1)

    def test_30_prevent_action_after_tool_budget_consumed(self):
        ctrl = self.ctrl(budget=SharedBudget(max_tool_calls=0))
        before = self.code.read_text()
        with self.assertRaises(RuntimeError):
            ctrl.observe_agent_action("edit_file", lambda: self.code.write_text("illegal"))
        self.assertEqual(self.code.read_text(), before)

    def test_28_observation_outcome_and_status_separate(self):
        led = self.ledger()
        r = led.record_check("x", "agent", lambda _: PASS)
        self.code.write_text("VALUE = 28\n")
        exported = led.export()
        self.assertEqual(exported["receipts"][0]["observed_outcome"], "PASS")
        self.assertEqual(exported["receipts"][0]["status"], "STALE")
        self.assertNotIn("PROTECTED_REFERENCE_PATCH", json.dumps(exported))

    def test_31_strict_manifest_requires_test_selection_collection_and_environment(self):
        incomplete = VisibleCheck("x", ("pytest", "-q", "tests/x.py"), "0" * 64)
        with self.assertRaises(ValueError):
            VisibleManifest((incomplete,), strict=True)
        self.assertTrue(VisibleManifest((self.check,), strict=True).strict)
        with self.assertRaises(ValueError):
            VisibleCheck("z", ("pytest", "-q"), "0"*64, working_dir="../outside")

    def test_32_expected_collection_mismatch_not_an_accepted_pass(self):
        led = self.ledger()
        obs = TestObservation(0, 2, 2, 0, 0, 0.1)
        receipt = led.record_check("x", "agent", lambda _: obs)
        self.assertEqual(receipt.observed_outcome, "PASS")
        self.assertEqual(receipt.qualification_error, "UNEXPECTED_COLLECTION")
        self.assertEqual(led.status(receipt), "ERROR")
        ctrl = self.ctrl(arm="B", executor=lambda _: obs)
        self.assertFalse(ctrl.final_gate().accepted)

    def test_33_collection_count_consistency_and_skipped(self):
        samples = [
            TestObservation(0, 1, 1, 0, 0, 0.1, tests_skipped=1),
            TestObservation(0, 2, 0, 0, 0, 0.1, tests_skipped=2),
            TestObservation(0, 2, 1, 0, 0, 0.1, tests_skipped=1),
        ]
        self.assertEqual([x.observed_outcome for x in samples], ["ERROR", "ERROR", "PASS"])

    def test_34_final_gate_patch_binding_and_post_gate_edit_rejection(self):
        ctrl = self.ctrl(arm="B")
        dec = ctrl.final_gate()
        self.assertTrue(dec.accepted)
        old_digest = dec.accepted_patch_sha256
        self.assertEqual(old_digest, hashlib.sha256(self.code.read_bytes()).hexdigest())
        self.code.write_text("VALUE = 99\n")
        self.assertNotEqual(old_digest, hashlib.sha256(self.code.read_bytes()).hexdigest())
        assessed = assess_attempt(AttemptRecord(
            "task", "B", 20261105, 1, "SUBMITTED", True, True,
            hashlib.sha256(self.code.read_bytes()).hexdigest(),
            dec.accepted_revision + 1, "PASS", dec))
        self.assertEqual(assessed.protected_verification, "PASS")
        self.assertFalse(assessed.verified_delivery)
        self.assertEqual(assessed.delivery_reason, "PATCH_OR_WORKSPACE_CHANGED_AFTER_GATE")

    def test_35_controller_test_mutates_production_patch_during_gate(self):
        ctrl = self.ctrl(arm="B", executor=lambda _: (
            self.code.write_text("VALUE = 42\n"), PASS)[1])
        dec = ctrl.final_gate()
        self.assertFalse(dec.accepted)

    def test_36_C_rechecks_bounded_and_manifest_ordered(self):
        manifest = VisibleManifest((self.check, self.check2), strict=True)
        ctrl = self.ctrl(manifest=manifest)
        ctrl.max_rechecks_per_edit = 1
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        ctrl.ledger.record_check("y", "agent", lambda _: PASS)
        out = ctrl.observe_agent_action("edit_file", lambda: self.code.write_text("VALUE = 7\n"))
        self.assertEqual(out['feedback']['rechecks'], [
            {"check_id": "x", "status": "CURRENT_PASS"},
            {"check_id": "y", "status": "UNVERIFIED_RECHECK_LIMIT"}])
        self.assertEqual(ctrl.ledger.current_status("y"), "STALE")
        self.assertEqual(ctrl.auto_rechecks_used, 1)

    def test_37_C_preserves_final_validation_reserve(self):
        budget = SharedBudget(max_wall_seconds=65, used_wall_seconds=5, final_reserve_seconds=60)
        ctrl = self.ctrl(budget=budget)
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        with self.assertRaises(RuntimeError):
            ctrl.observe_agent_action("edit_file", lambda: self.code.write_text("NOT ALLOWED\n"))
        self.assertEqual(ctrl.budget.used_test_runs, 0)
        self.assertEqual(self.code.read_text(), "VALUE = 1\n")

    def test_38_C_proactive_recheck_cannot_spend_final_reserve(self):
        budget = SharedBudget(max_wall_seconds=65, used_wall_seconds=4.8, final_reserve_seconds=60)
        ctrl = self.ctrl(budget=budget)
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        result = ctrl.observe_agent_action("edit_file", lambda: self.code.write_text("VALUE = 88\n"))
        self.assertEqual(result["feedback"]["rechecks"][0]["status"], "UNVERIFIED_BUDGET")
        self.assertEqual(ctrl.budget.used_test_runs, 0)
        self.assertEqual(ctrl.ledger.current_status("x"), "STALE")

    def test_39_protected_grade_separate_from_verified_delivery(self):
        digest = "f" * 64
        for term in ("TIMEOUT", "STOPPED", "GATE_REJECTED"):
            rec = AttemptRecord("task", "A", 20261105, 1, term,
                                False, True, digest, 1, "PASS")
            outcome = assess_attempt(rec)
            self.assertEqual(outcome.protected_verification, "PASS")
            self.assertFalse(outcome.verified_delivery)
        successful = AttemptRecord("task", "A", 20261105, 1,
                                   "SUBMITTED", True, True, digest, 1, "PASS")
        self.assertTrue(assess_attempt(successful).verified_delivery)

    def test_40_rejected_gate_can_still_have_protected_pass(self):
        rec = AttemptRecord("task", "C", 20261105, 1, "GATE_REJECTED", True,
                            True, "c"*64, 2, "PASS", GateDecision(False,"VISIBLE_CHECK_FAILED",{},1))
        result = assess_attempt(rec)
        self.assertEqual(result.protected_verification, "PASS")
        self.assertFalse(result.verified_delivery)
        self.assertEqual(result.delivery_reason, "NOT_DELIVERED")

    def test_41_no_patch_or_unexecuted_protected_check(self):
        rec = AttemptRecord("task", "B", 20261105, 1, "TIMEOUT", False,
                            False, None, 0, "NOT_RUN")
        res = assess_attempt(rec)
        self.assertEqual(res.protected_verification, "NOT_RUN")
        self.assertFalse(res.verified_delivery)

    def test_42_successful_gate_bound_to_exact_production_patch(self):
        ctrl = self.ctrl(arm="C")
        gate = ctrl.final_gate()
        digest = hashlib.sha256(self.code.read_bytes()).hexdigest()
        rec = AttemptRecord("task", "C", 20261105, 1, "SUBMITTED", True,
                            True, digest, gate.accepted_revision, "PASS", gate,
                            final_workspace_sha256=gate.accepted_workspace_sha256)
        self.assertTrue(assess_attempt(rec).verified_delivery)

    def test_43_attempt_registry_preserves_failures_without_replacing(self):
        reg = AttemptRegister()
        first = AttemptRecord("task", "B", 1, 1, "TIMEOUT", False, True,
                              "a"*64, 0, "FAIL")
        reg.append(first)
        second = AttemptRecord("task", "B", 1, 2, "SUBMITTED", True, True,
                               "b"*64, 1, "PASS")
        with self.assertRaises(ValueError):
            reg.append(second)
        self.assertEqual(reg.primary("task", "B", 1), first)

    def test_44_infra_retry_preserved_and_explicitly_selected(self):
        reg = AttemptRegister()
        first = AttemptRecord("task", "B", 1, 1, "INFRA_ERROR", False,
                              False, None, 0, "INFRA_ERROR",
                              environment_sha256="e"*64, configuration_sha256="c"*64)
        second = AttemptRecord("task", "B", 1, 2, "SUBMITTED", True,
                               True, "d"*64, 1, "PASS", infrastructure_retry_of=1,
                               environment_sha256="e"*64, configuration_sha256="c"*64,
                               retry_evidence_id="infra-case-001")
        reg.append(first)
        reg.append(second)
        self.assertEqual(reg.primary("task", "B", 1), second)
        self.assertEqual(len(reg.attempts), 2)

    def test_45_observed_regression_clock_from_first_invalidating_edit(self):
        ctrl = self.ctrl(executor=lambda _: FAIL)
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        ctrl.observe_agent_action("edit_file", lambda: self.code.write_text("VALUE = 15\n"))
        ev = [x for x in ctrl.ledger.events if x.get('type') == 'OBSERVED_PASS_TO_FAIL_AFTER_EDIT'][0]
        self.assertLessEqual(ev['invalidating_edit_observed_monotonic'],
                             ev['failure_observed_monotonic'])
        self.assertGreaterEqual(ev['edit_to_detection_seconds'], 0)

    def test_46_immutability_of_recorded_pass_outcome(self):
        led = self.ledger()
        previous = led.record_check("x", "agent", lambda _: PASS)
        self.code.write_text("VALUE = 6\n")
        led.observe_workspace("edit_file")
        self.assertEqual(previous.observed_outcome,"PASS")
        self.assertEqual(led.status(previous),"STALE")

    def test_47_gate_rechecks_after_late_workspace_mutation_without_patch_change(self):
        ctrl = self.ctrl(arm="B")
        old = ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        cfg = self.root / "runtime.cfg"
        cfg.write_text("A\n")
        # Re-run current as the new config itself made the original receipt stale.
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        original = ctrl.ledger.observe_workspace
        injected = [False]
        def late(action):
            if action == "final_gate_patch_sync" and not injected[0]:
                cfg.write_text("B\n")
                injected[0] = True
            return original(action)
        ctrl.ledger.observe_workspace = late
        decision = ctrl.final_gate()
        self.assertTrue(injected[0]); self.assertTrue(decision.accepted)
        self.assertEqual(decision.attempts, 1)
        self.assertNotEqual(old.workspace_sha256_at_end, decision.accepted_workspace_sha256)
        self.assertEqual(ctrl.ledger.current_status("x"), "CURRENT_PASS")
        self.assertEqual(decision.accepted_workspace_sha256, ctrl.ledger.snapshot.sha256)

    def test_48_gate_final_allowed_recheck_success_is_accepted(self):
        count = [0]
        config = self.root / "runtime.cfg"
        def run(_):
            count[0] += 1
            if count[0] < 3:
                config.write_text(str(count[0]))
            return PASS
        ctrl = self.ctrl(arm="B", executor=run)
        decision = ctrl.final_gate()
        self.assertEqual(count[0], 3)
        self.assertTrue(decision.accepted)
        self.assertEqual(decision.attempts, 3)

    def test_49_C_overrunning_proactive_test_cannot_qualify(self):
        budget = SharedBudget(max_wall_seconds=900, used_wall_seconds=839.3,
                              final_reserve_seconds=60)
        ctrl = self.ctrl(budget=budget, executor=lambda _: TestObservation(0,1,1,0,0,1.3))
        ctrl.ledger.record_check("x", "agent", lambda _: PASS)
        res = ctrl.observe_agent_action("edit_file", lambda: self.code.write_text("VALUE = 49\n"))
        self.assertEqual(res["feedback"]["rechecks"][0]["status"], "UNVERIFIED_BUDGET")
        self.assertEqual(ctrl.ledger.current_status("x"), "ERROR")
        self.assertGreater(budget.used_wall_seconds, 840.0) # honest elapsed accounting
        self.assertTrue(any(e.get("reason")=="ACTUAL_RUNTIME_EXCEEDED_ALLOWED_DEADLINE"
                            for e in ctrl.ledger.events))

    def test_50_no_patch_infra_failure_retains_classification(self):
        attempt = AttemptRecord("task", "B", 1, 1, "INFRA_ERROR", False,
                                False, None, 0, "INFRA_ERROR")
        assessment = assess_attempt(attempt)
        self.assertEqual(assessment.protected_verification, "INFRA_ERROR")
        self.assertTrue(assessment.infrastructure_fault)
        self.assertEqual(assessment.termination, "INFRA_ERROR")
        self.assertFalse(assessment.patch_present)

    def test_51_same_patch_digest_different_workspace_fails_delivery(self):
        ctrl = self.ctrl(arm="B")
        gate = ctrl.final_gate(); self.assertTrue(gate.accepted)
        cfg = self.root / "runtime.cfg"; cfg.write_text("B\n")
        ctrl.ledger.observe_workspace("out_of_band_edit")
        rec = AttemptRecord("task", "B", 1, 1, "SUBMITTED", True,
                            True, gate.accepted_patch_sha256, ctrl.ledger.revision,
                            "PASS", gate, final_workspace_sha256=ctrl.ledger.snapshot.sha256)
        assessment = assess_attempt(rec)
        self.assertFalse(assessment.verified_delivery)
        self.assertEqual(assessment.protected_verification, "PASS")
        self.assertEqual(assessment.delivery_reason, "PATCH_OR_WORKSPACE_CHANGED_AFTER_GATE")

    def test_52_missing_delivery_workspace_is_never_accepted(self):
        ctrl = self.ctrl(arm="B")
        gate = ctrl.final_gate()
        rec = AttemptRecord("task", "B", 1, 1, "SUBMITTED", True,
                            True, gate.accepted_patch_sha256, gate.accepted_revision,
                            "PASS", gate)
        self.assertFalse(assess_attempt(rec).verified_delivery)

    def test_53_retry_requires_same_config_env_and_evidence(self):
        first = AttemptRecord("task", "B", 1, 1, "INFRA_ERROR", False,
                              False, None, 0, "INFRA_ERROR",
                              environment_sha256="e"*64, configuration_sha256="c"*64)
        reg = AttemptRegister();reg.append(first)
        for env, cfg, evidence in (("e"*64,"d"*64,"case"),("f"*64,"c"*64,"case"),("e"*64,"c"*64,None)):
            with self.subTest(env=env, cfg=cfg, evidence=evidence):
                second = AttemptRecord("task", "B", 1, 2, "SUBMITTED", True,
                                       True, "b"*64, 1, "PASS", infrastructure_retry_of=1,
                                       environment_sha256=env, configuration_sha256=cfg,
                                       retry_evidence_id=evidence)
                with self.assertRaises(ValueError):reg.append(second)
        self.assertEqual(len(reg.attempts), 1)

    def test_54_last_edit_after_gate_stales_receipt_and_export_refreshes(self):
        ctrl = self.ctrl(arm="B")
        gate = ctrl.final_gate();self.assertTrue(gate.accepted)
        self.code.write_text("VALUE = 54\n")
        export = ctrl.ledger.export()
        self.assertNotEqual(export["workspace_sha256"],gate.accepted_workspace_sha256)
        self.assertEqual(export["receipts"][-1]["status"],"STALE")

    def test_55_B_C_identical_new_final_gate_after_workflow_change(self):
        for arm in ("B", "C"):
            with self.subTest(arm=arm):
                ctrl = self.ctrl(arm=arm)
                old = ctrl.ledger.record_check("x", "agent", lambda _: PASS)
                self.code.write_text(f"VALUE = {arm!r}\n")
                gate = ctrl.final_gate()
                self.assertTrue(gate.accepted)
                self.assertEqual(gate.attempts,1)
                self.assertEqual(ctrl.ledger.status(old),"STALE")

    def test_56_visible_budget_overrun_cannot_be_final_approval(self):
        budget = SharedBudget(max_wall_seconds=1.0, final_reserve_seconds=0)
        ctrl = self.ctrl(arm="B", budget=budget, executor=lambda _: TestObservation(0,1,1,0,0,1.1))
        decision = ctrl.final_gate()
        self.assertFalse(decision.accepted)
        self.assertEqual(decision.status,"UNVERIFIED_BUDGET")
        self.assertEqual(ctrl.ledger.current_status("x"),"ERROR")


if __name__ == "__main__":
    unittest.main(verbosity=2)


Writing test_gp02c_ledger.py


**Cell 02 observation criterion:** `test_gp02c_ledger.py` is created successfully. No real repository tests are invoked.

## 03 — Execute all contract fixtures

In [3]:
import sys, subprocess, re, hashlib, json
from pathlib import Path
r = subprocess.run([sys.executable, '-m', 'unittest', '-v', 'test_gp02c_ledger'],
                   capture_output=True, text=True, check=False)
print(r.stderr)
if r.stdout:
    print(r.stdout)
match = re.search(r'Ran (\d+) tests?', r.stderr)
assert r.returncode == 0 and match and 'OK' in r.stderr, 'CPU contract checks failed'
TEST_COUNT = int(match.group(1))
print(f'CONFIRMED: {TEST_COUNT} CPU-only contract checks passed; no GPU/model calls.')

test_01_pass_then_edit_stales_without_regression (test_gp02c_ledger.Fixtures.test_01_pass_then_edit_stales_without_regression) ... ok
test_02_post_edit_pass_new_receipt (test_gp02c_ledger.Fixtures.test_02_post_edit_pass_new_receipt) ... ok
test_03_pass_then_edit_then_fail_observed_regression (test_gp02c_ledger.Fixtures.test_03_pass_then_edit_then_fail_observed_regression) ... ok
test_04_rejected_or_noop_edit_does_not_advance_revision (test_gp02c_ledger.Fixtures.test_04_rejected_or_noop_edit_does_not_advance_revision) ... ok
test_05_shell_mutation_detected (test_gp02c_ledger.Fixtures.test_05_shell_mutation_detected) ... ok
test_06_test_itself_mutates_code_invalid_receipt (test_gp02c_ledger.Fixtures.test_06_test_itself_mutates_code_invalid_receipt) ... ok
test_07_zero_collection_is_not_pass (test_gp02c_ledger.Fixtures.test_07_zero_collection_is_not_pass) ... ok
test_08_timeout_error_malformed_results_and_skipped (test_gp02c_ledger.Fixtures.test_08_timeout_error_malformed_results_and_skip

**Cell 03 interpretation:** `OK` demonstrates only that the deterministic **controller contract checks** passed. It does *not* estimate Gemma performance, demonstrate benchmark success or establish novelty.

## 04 — Illustrate the arm separation with synthetic visible-test observations

All three arms begin with the same synthetic passing visible-test receipt at revision 0 and then experience one source edit. **A:** no final gate; **B:** one final recheck; **C:** one post-edit recheck and the identical B/C final gate. This scenario deliberately does not evaluate agent repair or protected tests.

In [4]:
from gp02c_ledger import (VisibleCheck, VisibleManifest, EvidenceLedger,
    TestObservation, VerificationController, SharedBudget, environment_fingerprint)
import tempfile

rows = []
for arm in ('A', 'B', 'C'):
    with tempfile.TemporaryDirectory(prefix='gp02c_demo_') as work:
        target = Path(work) / 'app.py'
        target.write_text('VALUE = 0\n')
        env = environment_fingerprint({'fixture': 'frozen-demo'})
        ck = VisibleCheck('visible_smoke', ('pytest', '-q', 'tests/test_visible_smoke.py'),
                          hashlib.sha256(b'frozen-visible-smoke').hexdigest(), reserve_seconds=0.1,
                          working_dir='.', test_selection='tests/test_visible_smoke.py',
                          expected_collected=1, pinned_environment_sha256=env)
        ledger = EvidenceLedger(work, VisibleManifest((ck,), strict=True), run_id='cpu-demo',
                                task_id='synthetic_case', arm=arm, seed=20261105,
                                environment_sha256=env)
        result = TestObservation(0, 1, 1, 0, 0, 0.2, stdout='synthetic pass')
        ledger.record_check('visible_smoke', 'common_initial', lambda c: result)
        budget = SharedBudget(max_wall_seconds=20, max_tool_calls=10, max_model_calls=10, final_reserve_seconds=5)
        controller = VerificationController(ledger, budget, lambda c: result,
                        production_patch_sha256=lambda: hashlib.sha256(target.read_bytes()).hexdigest())
        change = controller.observe_agent_action('edit_file',
                     lambda: target.write_text('VALUE = 1\n'), elapsed_seconds=0.5)
        mid_status = ledger.current_status('visible_smoke')
        gate = None if arm == 'A' else controller.final_gate()
        rows.append({'arm': arm, 'revision': ledger.revision,
                     'status_after_edit': mid_status,
                     'proactive_rechecks': len(change['feedback']['rechecks']),
                     'final_gate': 'NOT_APPLICABLE' if gate is None else gate.status,
                     'controller_accepted': None if gate is None else gate.accepted,
                     'charged_tool_calls_after_edit_and_gate': budget.used_tool_calls,
                     'charged_visible_tests': budget.used_test_runs})
for x in rows:
    print(x)
assert rows[0]['proactive_rechecks'] == 0
assert rows[1]['proactive_rechecks'] == 0 and rows[1]['controller_accepted']
assert rows[2]['proactive_rechecks'] == 1 and rows[2]['controller_accepted']
print('CONFIRMED: B and C share the final gate; C alone proactively rechecks after an edit.')

{'arm': 'A', 'revision': 1, 'status_after_edit': 'STALE', 'proactive_rechecks': 0, 'final_gate': 'NOT_APPLICABLE', 'controller_accepted': None, 'charged_tool_calls_after_edit_and_gate': 1, 'charged_visible_tests': 0}
{'arm': 'B', 'revision': 1, 'status_after_edit': 'STALE', 'proactive_rechecks': 0, 'final_gate': 'VERIFIED_VISIBLE', 'controller_accepted': True, 'charged_tool_calls_after_edit_and_gate': 2, 'charged_visible_tests': 1}
{'arm': 'C', 'revision': 1, 'status_after_edit': 'CURRENT_PASS', 'proactive_rechecks': 1, 'final_gate': 'VERIFIED_VISIBLE', 'controller_accepted': True, 'charged_tool_calls_after_edit_and_gate': 2, 'charged_visible_tests': 1}
CONFIRMED: B and C share the final gate; C alone proactively rechecks after an edit.


**Cell 04 interpretation:** The three arms exhibit the intended *policy* differences on controlled, identical synthetic receipts. The table is not a task-level effect estimate; real success remains determined by later independent protected verification.

## 05 — Export reproducible CPU validation report and compact evidence ZIP

In [5]:
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED
files = [Path('gp02c_ledger.py'), Path('test_gp02c_ledger.py')]
report = {
    'stage': 'GP02C-02', 'version': '1.2', 'kind': 'CPU-only deterministic contract validation',
    'executed_utc': datetime.now(timezone.utc).isoformat(),
    'tests_run': TEST_COUNT, 'tests_passed': TEST_COUNT,
    'tests_failed': 0, 'unit_test_return_code': r.returncode,
    'synthetic_arm_illustration': rows,
    'sha256': {f.name: hashlib.sha256(f.read_bytes()).hexdigest() for f in files},
    'research_guardrails': {
        'model_calls': 0, 'gpu_use': False,
        'protected_evaluator_used': False, 'held_out_36_tasks_used': False,
        'gp02b_v27_modified': False,
        'actual_protected_repair_outcomes': 'NOT_MEASURED',
        'live_worker_integration': 'NOT_IMPLEMENTED',
        'hard_process_deadline_enforcement': 'NOT_IMPLEMENTED',
        'isolated_test_snapshot': 'NOT_IMPLEMENTED'
    }
}
report_file = Path('GP02C_02_CPU_Validation_Results_v1_2.json')
report_file.write_text(json.dumps(report, indent=2) + '\n')
zip_file = Path('GP02C_02_CPU_Evidence_v1_2.zip')
with ZipFile(zip_file, 'w', ZIP_DEFLATED) as archive:
    for f in files + [report_file]:
        archive.write(f, f.name)
print(f'CPU validation report: {report_file} ({report_file.stat().st_size} bytes)')
print(f'CPU evidence archive: {zip_file} ({zip_file.stat().st_size} bytes)')
print('Manifest SHA-256:', hashlib.sha256(report_file.read_bytes()).hexdigest())

CPU validation report: GP02C_02_CPU_Validation_Results_v1_2.json (1729 bytes)
CPU evidence archive: GP02C_02_CPU_Evidence_v1_2.zip (16644 bytes)
Manifest SHA-256: 0de8832b13eae941250a2176c29f51832b50876fe1c782cea41f4273985349cc


## Interpretation and next stage: GP02C-03 — Trusted live adapter

A passing synthetic CPU suite establishes contract compliance for the tested cases, **not** live repair reliability. Before running Gemma: build a real isolated-source `pytest`/JUnit adapter, use enforceable monotonic deadlines and process-group termination at 840-second repair / 900-second overall cutoffs, check that all budget counts are charged exactly once, capture immutable patch/workspace identity, score final production snapshots independently, and complete the six-trace evidence-exposure audit.

Only after a small paired *development-only* B/C pilot should the held-out cohort and evaluation budgets be frozen. The paper's main C−B claim is **not yet tested**.
